### 1. Loading the VectorDB

In [1]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
def load_vectorDB():
    embedding_model=HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )


    vectorstore = Chroma(
        persist_directory='chroma_langchain_db',
        embedding_function=embedding_model
    )
    return vectorstore


### 2. Making the Retriever

In [2]:
def build_retriever(vectorstore):
    retriever=vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 6, "fetch_k": 20}
    )
    print(f"Retriever Ready (top_k={4})")
    return retriever

### 3. Build LLM

In [3]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
def build_llm():
    llm=ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2,
        max_tokens=1024,
        api_key=os.getenv("GROQ_API_KEY")
    )
    print(f" Groq LLM is ready ")
    return llm

### 3. Make Prompt Template

In [ ]:
from langchain_core.prompts import PromptTemplate
def build_prompt():
    template = """
You are an intelligent AI tutor helping students understand concepts clearly.You should know what is the pdf talking about
and also answer the user queries with the help of the context provided following the given guidelines.


Guidelines for your answer:
- Start with a clear, direct definition or answer in 1-2 sentences
- Then elaborate with supporting details, types, or examples from the context
- Use bullet points or numbered lists where the context provides them
- If the context contains a comparison table or algorithm properties, include it
- Keep the answer concise but complete — do not pad unnecessarily
- If the context does not contain enough information, use your external knowledge to answer that


Context:
{context}

Question: {question}

Answer:

"""
    return PromptTemplate.from_template(template)

### Ask questions to the complete RAG chain

In [8]:
def ask_question(retriever,prompt,llm,question):
    print(f"\n Your Question: {question}")
    print("-"*50)

    docs=retriever.invoke(question)
    context="\n\n".join([doc.page_content for doc in docs])
    print(context)

    formatted_prompt=prompt.invoke({
        "context":context,
        "question":question
    })
    response=llm.invoke(formatted_prompt)
    print(f"Answer:\n{response.content}")
    print("\n Source Chunks used:")
    # for i,doc in enumerate(docs,1):
    #     source=doc.metadata.get("source","unknown")
    #     page=doc.metadata.get("page","?")

    #     print(f"[{i}] {source} (page {page})")
    #     print(f"      {doc.page_content[:150]}...\n")

In [9]:
if __name__=="__main__":
    vectorstore=load_vectorDB()
    retriever=build_retriever(vectorstore)
    llm=build_llm()
    prompt=build_prompt()

    print("="*50)
    print(" RAG pipeline Ready! Ask questions")
    print("Type 'exit' to Quit")
    print("="*50)

    while True:
        user_input=input("\nYou: ").strip()
        if(user_input.lower() in ["exit","quit","q"]):
            print("👋 Bye!")
            break
        if not user_input:
            continue

        ask_question(retriever,prompt,llm,user_input)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Retriever Ready (top_k=4)
 Groq LLM is ready 
 RAG pipeline Ready! Ask questions
Type 'exit' to Quit

 Your Question: what is this pdf all about
--------------------------------------------------
technique used for partitioning data into K clusters based on similarity. It aims to 
minimize the within-cluster variance while maximizing the between-cluster 
variance, effectively grouping data points into clusters with similar characteristics.
Algorithm:
 Initialization:
Randomly initialize K cluster centroids (points in the feature space).
Alternatively, select K data points from the dataset as initial centroids.
 Assignment Step:
Assign each data point to the nearest cluster centroid based on a distance 
metric (commonly Euclidean distance).
Update the cluster assignments based on the new centroid positions.
 Update Step:
Recalculate the centroids of the clusters by computing the mean of all data 
points assigned to each cluster.
The new centroids represent the center of mass of th